In [2]:
import torch
import matplotlib.pyplot as plt
import cv2 as cv
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets,transforms
from torch.utils.data import random_split,DataLoader, ConcatDataset
import numpy as np

In [3]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

In [52]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
    def __init__(self):
        self.transform = transforms.Compose([
                                            transforms.Grayscale(num_output_channels=3),
                                            transforms.Resize((128,128)),
                                            transforms.ToTensor(),
                                            transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
                                        ])
        self.lane_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),  # binary mask, 1 channel
            transforms.Resize((128, 128)),
            transforms.ToTensor(),  # already scales to [0, 1]
        ])
        self.road_images = datasets.ImageFolder(root="/content/drive/MyDrive/road_lane_dir/road",transform=self.transform)
        self.lane_images = datasets.ImageFolder(root="/content/drive/MyDrive/road_lane_dir/lane",transform= self.lane_transform)
    def __len__(self):
        return min(len(self.road_images), len(self.lane_images))

    def __getitem__(self,index):
        images,_=self.road_images[index]
        lane,_=self.lane_images[index]
        return images,lane

In [5]:
dataset =CustomDataset()
print(f"Road images: {len(dataset.road_images)}")
print(f"Lane images: {len(dataset.lane_images)}")

Road images: 4930
Lane images: 4760


In [6]:
dataloader = DataLoader(dataset,batch_size=64,shuffle=True)

In [56]:
# def image_with_lane(img):
#     image = cv.imread(img,cv.IMREAD_GRAYSCALE)

#     con = cv.Canny(image, 50,150)

#     ret, threshold = cv.threshold(image, 150,255,cv.THRESH_BINARY)
#     cv2_imshow(image)
#     cv2_imshow(con)
#     cv2_imshow(threshold)

In [57]:
# data_uint8 = (king * 255).astype(np.uint8)
# img = Image.fromarray(data_uint8)
# img.save('output_image.png')


In [58]:
# image_with_lane(path)

In [7]:
class CustomModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = self.conv(3,32)
        self.conv2 = self.conv(32,64)
        self.conv3 = self.conv(64,128)

        self.up1 = nn.ConvTranspose2d(128,64,2,stride=2)
        self.up2 = nn.ConvTranspose2d(64,32,2,stride=2)
        self.up3 = nn.ConvTranspose2d(32,1,2,stride=2)
    def conv(self,inp_c, out_c):
        return nn.Sequential(
            nn.Conv2d(inp_c,out_c,3,padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

    def forward(self, x):

        cnn_out = self.conv3(self.conv2(self.conv1(x)))

        y = self.up3(self.up2(self.up1(cnn_out)))
        return y


In [8]:
model = CustomModel()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [9]:
model.load_state_dict(torch.load("lane_detection_hehe.pt"))

<All keys matched successfully>

In [10]:
#  train loop
epochs =2
from tqdm import tqdm
model.train()
for epoch in tqdm(range(epochs)):
    total_loss =0
    for image, lane in dataloader:
        image, lane = image.to(device), lane.to(device)

        outputs = model(image)

        loss = criterion(outputs, lane)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
    print(f"epochs {epoch} / {epochs} ====>>>. loss : {total_loss/len(dataloader)} ")


 50%|█████     | 1/2 [32:09<32:09, 1929.14s/it]

epochs 0 / 2 ====>>>. loss : 0.11197507858276368 


100%|██████████| 2/2 [35:51<00:00, 1075.64s/it]

epochs 1 / 2 ====>>>. loss : 0.10552813162406285 


In [11]:
model.eval()

CustomModel(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (up1): ConvTranspose2d(128, 64, kernel_size=(2, 2), stride=(2, 2))
  (up2): ConvTranspose2d(64, 32, kernel_size

In [13]:
torch.save(model.state_dict(),"lane_detection_hehe_1.pt")